# 02 — Volatility Lab: HV vs IV, Skew, and Term Structure

You will:
1. Compute **historical volatility** from a synthetic (seeded) price series.
2. Round-trip **implied volatility** with `pricing.implied_vol` / `pricing.bsm_price`.
3. Plot **skew** (IV vs strike) and **term structure** (IV vs DTE) from the sample chains.
4. Compare the vol regimes of **DEMO**, **LOWVOL**, and **HIGHVOL**.

All offline. DEMO spot **$100**, IV ~**25%**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import pricing, analyzer, data

## 1. Historical volatility from a synthetic price series

Simulate a seeded geometric random walk with a known 'true' vol, then estimate HV back out of
the daily log returns: `std(log returns) * sqrt(252)`. The estimate should land near the input.

In [ ]:
rng = np.random.default_rng(42)
true_vol, days, S0 = 0.25, 252, 100.0
daily = rng.normal(0, true_vol / np.sqrt(252), days)   # daily log returns
prices = S0 * np.exp(np.cumsum(daily))
log_ret = np.diff(np.log(prices))
hv = log_ret.std(ddof=1) * np.sqrt(252)
print(f'input (true) vol: {true_vol:.3f}   estimated HV: {hv:.3f}')

HV is a *backward* measurement of realized movement. Next: IV, the *forward* number the market
prices into options.

## 2. Implied vol round-trip

IV is the vol that reproduces the market price. Back it out of the DEMO ATM call mid (3.91), then
feed it back into `bsm_price` and confirm you recover 3.91 — `implied_vol` and `bsm_price` are
inverses.

In [ ]:
mid = 3.91
iv = pricing.implied_vol('call', price=mid, spot=100, strike=100, t=45/365)
back = pricing.bsm_price('call', 100, 100, 45/365, iv)
print(f'implied vol: {iv:.4f}')
print(f'reprice at that IV: {back:.2f}  (recovers the input {mid})')

## 3. Skew: IV varies across strikes

Load DEMO, take the 45-DTE rows, and plot the chain's own `iv` column against strike. Equity
names show a **put skew**: OTM puts (low strikes) carry higher IV than OTM calls (high strikes).

In [ ]:
demo = data.load_sample_chain('DEMO')
c45 = demo[(demo.kind == 'call') & (demo.expiry_days == 45)].sort_values('strike')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(c45.strike, c45.iv, 'o-')
ax.axvline(100, color='k', ls='--', lw=1, label='spot=100')
ax.set_xlabel('strike'); ax.set_ylabel('implied vol'); ax.set_title('DEMO 45-DTE skew'); ax.legend()
plt.show()

The curve slopes **down to the right**: lower strikes (downside puts) are bid up on crash-risk
and protection demand. That is why put-side premium selling collects more than the call side.

## 4. Term structure: ATM IV across expirations

Now hold the strike near ATM (100) and plot IV vs DTE. Upward slope = contango (calm);
downward = backwardation (near-term stress). DEMO is mildly backwarded.

In [ ]:
atm = demo[(demo.kind == 'call') & (demo.strike == 100)].sort_values('expiry_days')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(atm.expiry_days, atm.iv, 's-')
ax.set_xlabel('DTE'); ax.set_ylabel('ATM implied vol'); ax.set_title('DEMO ATM term structure')
plt.show()
print(atm[['expiry_days', 'iv']].to_string(index=False))

## 5. Compare three vol regimes

DEMO (~25%), LOWVOL (~14%), HIGHVOL (~55%). Pull each chain's ATM IV to see how different
'normal' is for different underlyings — the reason an absolute IV number means nothing.

In [ ]:
def atm_iv(name):
    ch = data.load_sample_chain(name)
    spot = ch.spot.iloc[0]
    calls = ch[ch.kind == 'call'].copy()
    row = calls.iloc[(calls.strike - spot).abs().argmin()]
    return spot, row.strike, row.iv

for nm in ['LOWVOL', 'DEMO', 'HIGHVOL']:
    spot, k, iv = atm_iv(nm)
    print(f'{nm:8s} spot {spot:6.1f}  ATM strike {k:6.1f}  ATM IV {iv:.3f}')

## 6. Skew, side by side

Overlay the three chains' skews (normalize the x-axis to strike/spot so they line up). HIGHVOL
sits far above; LOWVOL far below — same shape, very different levels.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for nm in ['LOWVOL', 'DEMO', 'HIGHVOL']:
    ch = data.load_sample_chain(nm)
    spot = ch.spot.iloc[0]
    dte = sorted(ch.expiry_days.unique())[len(ch.expiry_days.unique())//2]
    sl = ch[(ch.kind == 'call') & (ch.expiry_days == dte)].sort_values('strike')
    ax.plot(sl.strike / spot, sl.iv, 'o-', label=f'{nm} ({dte}d)')
ax.axvline(1.0, color='k', ls='--', lw=1)
ax.set_xlabel('strike / spot (moneyness)'); ax.set_ylabel('IV'); ax.set_title('Skew across vol regimes'); ax.legend()
plt.show()

## 7. Expected (implied) move

`analyzer.expected_move(spot, vol, t)` returns the 1-sigma move `spot * vol * sqrt(t)` — your
ruler for 'how far is the market pricing?' Compare DEMO's 45-DTE implied move at 25% vs a 55% event vol.

In [ ]:
em_calm  = analyzer.expected_move(100, 0.25, 45/365)
em_event = analyzer.expected_move(100, 0.55, 45/365)
print(f'1-sigma move @25% IV: +/- {em_calm:.2f}  (to ~{100-em_calm:.0f}/{100+em_calm:.0f})')
print(f'1-sigma move @55% IV: +/- {em_event:.2f}  (to ~{100-em_event:.0f}/{100+em_event:.0f})')

## Experiments

1. In section 1, change the seed and `true_vol` (try 0.10 and 0.60). Does the HV estimate track
   the input? Increase `days` to 1000 — does the estimate tighten?
2. In section 2, back out IV from the **95 put** mid (1.58) and the **110 call** mid (0.73).
   Do the per-strike IVs match the chain's `iv` column (the skew)?
3. In section 3, redo the skew with **puts** instead of calls. Same shape? (It should be — IV is
   a property of the strike, not the option kind, under one model.)
4. In section 4, build the term-structure plot for **HIGHVOL** (spot 62). Is it contango or
   backwardation, and what would that imply about near-term event risk?
5. In section 7, at what IV does the 45-DTE 1-sigma move reach 10 points? Solve by trying values —
   this is the 'implied move' a straddle buyer must beat.